# How AI Agents Think and Act

## Course Plan:

- Lecture 1 — From LLM to Agent: The Simplest Possible Loop — DONE
- Lecture 2 — Memory and RAG — DONE
- Lecture 3 — Graphs and Planning — DONE
- **Lecture 4 — Multi-Agent Systems** ← you are here

---

Prerequisites:
- `.env` file with `OPENAI_API_KEY` and `TAVILY_API_KEY`
- `pip install langgraph tavily-python openai python-dotenv`

# Lecture 4 — Multi-Agent Systems: Division of Labor

**Central question:** When does one agent become multiple agents, and why?

- Lectures 1–3 built progressively smarter *single* agents: a bare loop, a memory-augmented loop, and a graph-structured planner. Each improvement made the agent more capable — but all the intelligence still lived in one process, one prompt, one loop.
- Today we ask: *what happens when a task is too large, too varied, or too conflicted for one agent to handle well?*

The answer is **decomposition**: an **orchestrator** agent that plans and assigns work, and **worker** agents that execute specialized tasks. The interface between them — the thing that replaces function signatures and type systems — is a **natural language task description**.

> *"The interface between agents is a sentence."* — This is the payoff of the whole course.

---
# Setup

In [ ]:
# Standard imports and path setup
import os, sys, json
from typing import TypedDict, Literal, Annotated
import operator
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

# LangGraph core
from langgraph.graph import StateGraph, END
from IPython.display import Image, display

# API clients
from openai import OpenAI
from tavily import TavilyClient

openai_client: OpenAI = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
tavily_client: TavilyClient = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

print("Setup complete.")

---
# Part 1 — Why One Agent Is Not Always Enough

Before building a multi-agent system, we should understand what problem it solves. Let's look at three failure modes that appear when a single agent is asked to do too much.

In [ ]:
# Demonstrate role conflict: one prompt forced to be critic AND advocate simultaneously
conflicted_prompt: str = (
    "You are a helpful assistant. Write a persuasive argument that remote work "
    "increases productivity. Then critique that argument as a skeptic. "
    "Then recommend a policy based on both perspectives."
)

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": conflicted_prompt}],
    temperature=0.7,
)
output: str = response.choices[0].message.content or ""
print(output[:800])
print("\n... (output continues)")

<details>
<summary>Details: <strong>Three failure modes of single-agent systems</strong></summary>

**1. Role conflict.** One model cannot genuinely hold two opposing perspectives at once. Ask it to argue both sides and it hedges, averages, or drifts toward one position. A human team assigns "devil's advocate" to a different person for a reason.

**2. Context window exhaustion.** A single agent accumulates all context in one conversation: tools, results, reasoning, history. Complex tasks generate thousands of tokens of intermediate state. The model's attention degrades at long contexts; older information gets "forgotten" in practice even when it's technically present.

**3. No parallelism.** A single agent runs one step at a time. If three sub-tasks are independent, it still executes them serially. A multi-agent system can dispatch them simultaneously.

All three are arguments for **decomposition**: break the task, assign the pieces, recombine the results.
</details>

---
# Part 2 — The Orchestrator / Worker Pattern

The simplest multi-agent architecture has two roles:

- **Orchestrator** — receives the goal, decomposes it into tasks, assigns each task to a worker, and synthesizes results.
- **Worker** — receives one task (a natural language description), executes it using available tools, and returns a result.

The orchestrator never executes tasks directly. The worker never sees the big picture.

```
User Goal
    │
    ▼
[Orchestrator] ──task description──▶ [Worker A]
               ──task description──▶ [Worker B]
               ──task description──▶ [Worker C]
                        ◀── results ──
    │
    ▼
Final Answer
```

We will build this as a LangGraph — because the orchestrator and workers are nodes, and the task descriptions flowing between them are strings in the state dict.

In [ ]:
# The shared state that flows through the entire multi-agent graph
class MultiAgentState(TypedDict):
    goal: str                           # user's original request
    tasks: list[str]                    # orchestrator-generated task list (natural language)
    results: Annotated[list[str], operator.add]  # worker results, accumulated
    final_answer: str                   # synthesized output

print("MultiAgentState defined.")
print("Fields:", list(MultiAgentState.__annotations__.keys()))

<details>
<summary>Details: <strong>What is Annotated[list[str], operator.add]?</strong></summary>

Normal LangGraph state merging replaces a field's value with whatever the node returns. For a list that accumulates results from multiple workers, you want **appending**, not replacing.

`Annotated[list[str], operator.add]` tells LangGraph to use `operator.add` (list concatenation) as the merge strategy for the `results` field. When a worker returns `{"results": ["one finding"]}`, LangGraph appends it to the existing list rather than overwriting it.

This is LangGraph's **reducer** mechanism. Every field in state can have a custom reducer — the default reducer is "replace". Providing `operator.add` makes `results` an append-only accumulator, which is the right shape for collecting independent worker outputs.
</details>

---
# Part 3 — The Orchestrator Node

The orchestrator receives the user's goal and produces a list of tasks. Each task is a self-contained natural language instruction — enough for a worker to execute it without knowing the overall goal.

In [ ]:
# Orchestrator node: decompose goal into a list of research tasks
def orchestrator_node(state: MultiAgentState) -> dict:
    goal: str = state["goal"]
    print(f"[orchestrator] decomposing goal: '{goal}'")

    # Prompt the model to produce a numbered task list
    system_prompt: str = (
        "You are a research coordinator. Given a research goal, decompose it into "
        "exactly 3 specific, self-contained research tasks. "
        "Each task should be answerable by searching the web independently. "
        "Output ONLY a JSON array of 3 strings, like: "
        '[\"task one\", \"task two\", \"task three\"]. '
        "No other text."
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Research goal: {goal}"},
        ],
        temperature=0.2,
    )
    raw: str = (response.choices[0].message.content or "").strip()

    # Parse the JSON task list — these are the natural language instructions to workers
    tasks: list[str] = json.loads(raw)
    print(f"[orchestrator] generated {len(tasks)} tasks:")
    for i, task in enumerate(tasks, 1):
        print(f"  {i}. {task}")

    return {"tasks": tasks}

print("orchestrator_node defined.")

<details>
<summary>Details: <strong>The task list is the API</strong></summary>

In classical software, a function call is the unit of decomposition. The caller passes typed arguments; the callee's signature enforces the contract.

Here the orchestrator emits a list of strings — natural language task descriptions. The "API" is the prompt. The "type system" is the phrase "self-contained research task."

This is both the power and the risk: the orchestrator can generate any decomposition, including ambiguous or contradictory tasks. There is no compiler to catch "task 2 depends on the result of task 3." The contract is enforced by the model's judgment alone.

The JSON wrapper is a thin layer of structure — enough to parse reliably, but the content of each string is still free-form language.
</details>

---
# Part 4 — The Worker Node

Each worker receives a single task description and executes it: search for information, then write a concise answer. The worker has no knowledge of the other tasks or the overall goal.

In [ ]:
# Worker function: given a task description, search and produce a result string
def run_worker(task: str, worker_id: int) -> str:
    print(f"[worker-{worker_id}] executing: '{task[:60]}...'")

    # Step 1: search for relevant information
    search_response: dict = tavily_client.search(query=task, max_results=4)
    results: list[dict] = search_response.get("results", [])
    search_text: str = "\n".join(r.get("content", "") for r in results)

    if not search_text.strip():
        return f"[worker-{worker_id}] No results found for: {task}"

    # Step 2: summarize findings as a self-contained paragraph
    prompt: str = (
        f"You are a research assistant. Answer the following research task "
        f"concisely (2–3 sentences) based on the search results provided.\n\n"
        f"Task: {task}\n\n"
        f"Search results:\n{search_text[:2000]}\n\n"
        f"Answer:"
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    answer: str = (response.choices[0].message.content or "").strip()
    result: str = f"[Task: {task}]\n{answer}"
    print(f"[worker-{worker_id}] done. ({len(result)} chars)")
    return result

print("run_worker helper defined.")

In [ ]:
# Worker node: runs all tasks sequentially and appends results to state
def worker_node(state: MultiAgentState) -> dict:
    tasks: list[str] = state["tasks"]
    results: list[str] = []

    # Execute each task independently — same worker logic, different task string
    for i, task in enumerate(tasks, 1):
        result: str = run_worker(task, i)
        results.append(result)

    # Return list — Annotated reducer will append these to any existing results
    return {"results": results}

print("worker_node defined.")

<details>
<summary>Details: <strong>Worker isolation and specialization</strong></summary>

The worker function receives only a task string — it has no access to the goal, the other tasks, or the other workers' results. This isolation is intentional.

**Benefits:**
- Each worker can be independently tested, replaced, or specialized.
- A worker failure affects only its result, not the whole run.
- Workers can run in parallel (we will do this in Part 6).

**Trade-off:**
- A worker cannot ask "wait, does this conflict with what worker 2 found?" — cross-task consistency is the synthesizer's job.
- If task 2 genuinely depends on the output of task 1, this architecture breaks. Dependency between tasks requires a more complex routing structure (or a single sequential agent).

The design rule: orchestrator tasks should be **independent**. If they aren't, you need a different pattern.
</details>

---
# Part 5 — The Synthesizer Node

The synthesizer receives all worker results and produces a single coherent answer. This is the only node that sees both the original goal and everything the workers found.

In [ ]:
# Synthesizer node: combine all worker results into a final coherent answer
def synthesizer_node(state: MultiAgentState) -> dict:
    goal: str = state["goal"]
    results: list[str] = state["results"]
    print(f"[synthesizer] combining {len(results)} worker results")

    # Format all results as a numbered list for the model
    combined_results: str = "\n\n".join(
        f"Finding {i}:\n{r}" for i, r in enumerate(results, 1)
    )

    # Prompt sees the original goal and all worker outputs — nothing else
    prompt: str = (
        f"You are a senior research analyst. The following findings were gathered "
        f"by independent research workers to address a research goal. "
        f"Synthesize them into a single, well-structured answer (4–6 sentences).\n\n"
        f"Original goal: {goal}\n\n"
        f"Worker findings:\n{combined_results}\n\n"
        f"Synthesized answer:"
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    final_answer: str = (response.choices[0].message.content or "").strip()
    print(f"[synthesizer] answer ready ({len(final_answer)} chars)")
    return {"final_answer": final_answer}

print("synthesizer_node defined.")

<details>
<summary>Details: <strong>Why a separate synthesizer?</strong></summary>

The synthesizer could be eliminated — the orchestrator could collect results and write the final answer itself. But giving it a dedicated node has two advantages:

1. **Separation of concerns.** Decomposition (orchestrator) and integration (synthesizer) are cognitively distinct tasks. A model prompted for one does it better than a model prompted for both.

2. **Inspectability.** The synthesizer receives all worker results as an explicit, readable list in the state dict. At any point you can inspect `state["results"]` to see exactly what the synthesizer was working from — no hidden context.

The synthesizer is the only node in this graph that sees the full picture. Everything before it is specialized. This mirrors how research teams work: analysts gather, editors synthesize.
</details>

---
# Part 6 — Assemble and Run the Multi-Agent Graph

Wire the three nodes into a graph: orchestrator → worker → synthesizer.

In [ ]:
# Assemble the multi-agent graph
ma_builder: StateGraph = StateGraph(MultiAgentState)

# Register nodes
ma_builder.add_node("orchestrator", orchestrator_node)
ma_builder.add_node("worker", worker_node)
ma_builder.add_node("synthesizer", synthesizer_node)

# Wire: orchestrator → worker → synthesizer → END
ma_builder.set_entry_point("orchestrator")
ma_builder.add_edge("orchestrator", "worker")
ma_builder.add_edge("worker", "synthesizer")
ma_builder.add_edge("synthesizer", END)

# Compile and render
multi_agent_graph = ma_builder.compile()
display(Image(multi_agent_graph.get_graph().draw_mermaid_png()))

<details>
<summary>Details: <strong>Reading the diagram</strong></summary>

The graph is a simple chain: orchestrator → worker → synthesizer → END. There is no branching, no loop. This is a pipeline — three stages, each dependent on the previous one's output.

Compare to the research graph in Lecture 3, which had a loop with a conditional retry. The multi-agent graph is simpler in structure but richer in semantics: each node runs a different *agent* with a different *role*, not just a different function.

In Lecture 3, one agent graph controlled the loop. Here, multiple agents collaborate in a linear pipeline. The next step (Part 7) introduces branching back: what if a worker fails?
</details>

In [ ]:
# Run the full multi-agent system on a research goal
GOAL: str = "How does gradient descent work and why does it sometimes get stuck in local minima?"

initial_state: MultiAgentState = {
    "goal": GOAL,
    "tasks": [],
    "results": [],
    "final_answer": "",
}

print(f"Goal: {GOAL}\n")
final_state: MultiAgentState = multi_agent_graph.invoke(initial_state)

print("\n--- Final Answer ---")
print(final_state["final_answer"])

In [ ]:
# Inspect the intermediate state — tasks and worker results
print("Tasks generated by orchestrator:")
for i, task in enumerate(final_state["tasks"], 1):
    print(f"  {i}. {task}")

print(f"\nWorker results ({len(final_state['results'])} total):")
for i, result in enumerate(final_state["results"], 1):
    print(f"\n  --- Result {i} ---")
    print(f"  {result[:300]}")

---
# Part 7 — The Interface Between Agents Is a Sentence

Pause here and look at what actually flowed between the orchestrator and the workers. The tasks list contains plain English sentences. There is no schema, no type, no contract — just a string that a worker must interpret and execute.

This is the moment the course has been building toward.

In [ ]:
# Show exactly what crossed the agent boundary
print("=== The interface between orchestrator and workers ===\n")
print("Type:", type(final_state["tasks"]))
print("Element type:", type(final_state["tasks"][0]))
print()
for i, task in enumerate(final_state["tasks"], 1):
    print(f"Message {i} (a plain string):")
    print(f"  \"{task}\"")
    print()
print("This string is the entire contract between the orchestrator and worker {i}.")
print("No schema. No type enforcement. No compiler check.")
print("The worker interprets it with a language model.")

<details>
<summary>Details: <strong>Language as API — what this buys and what it costs</strong></summary>

**What it buys:**
- **Flexibility.** The orchestrator can decompose any goal into any task, without registering new function signatures. A new kind of task needs no code change — just a different sentence.
- **Generality.** Workers built for one domain can often handle adjacent domains, because natural language generalizes. A "research assistant" worker can follow instructions about machine learning, history, or cooking.
- **Evolvability.** You can change what the orchestrator requests by changing its prompt, not its code.

**What it costs:**
- **Brittleness.** A badly-phrased task produces a bad result with no error. The worker does not say "I don't understand" — it tries to comply and may produce confident nonsense.
- **No contract enforcement.** In classical software, if the API signature changes, the caller breaks at compile time. In a language-interface system, the caller (orchestrator) silently starts sending tasks the worker misunderstands.
- **Debugging is hard.** When a worker produces a wrong result, is the fault in the orchestrator's task description, the worker's prompt, the search results, or the synthesis? All of these are strings. None of them have a stack trace.

This trade-off — flexibility at the cost of reliability — is the central tension in all current agent systems.
</details>

---
# Part 8 — Specialized Workers

So far all workers run the same function. The power of multi-agent systems grows when workers are **specialized** — each built with a different prompt, different tools, or a different model.

Let's build a system with two distinct worker types: a **researcher** (uses Tavily to find facts) and a **critic** (uses only the model to find weaknesses and counterarguments).

In [ ]:
# State for the specialized worker system
class SpecializedState(TypedDict):
    claim: str                                   # the claim to investigate
    research: str                                # factual findings from the researcher
    critique: str                                # weaknesses found by the critic
    verdict: str                                 # final balanced assessment

# Researcher worker: search for evidence supporting or explaining the claim
def researcher_node(state: SpecializedState) -> dict:
    claim: str = state["claim"]
    print(f"[researcher] investigating: '{claim[:60]}...'")

    # Search for factual information about the claim
    search_response: dict = tavily_client.search(query=claim, max_results=5)
    results: list[dict] = search_response.get("results", [])
    search_text: str = "\n".join(r.get("content", "") for r in results)

    # Summarize what the evidence actually says
    prompt: str = (
        f"You are a careful researcher. Based on the search results, summarize "
        f"what evidence exists about the following claim. Be factual and neutral.\n\n"
        f"Claim: {claim}\n\nSearch results:\n{search_text[:2000]}\n\nResearch summary:"
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    research: str = (response.choices[0].message.content or "").strip()
    print(f"[researcher] done ({len(research)} chars)")
    return {"research": research}

# Critic worker: find weaknesses, counterarguments, and caveats — no search
def critic_node(state: SpecializedState) -> dict:
    claim: str = state["claim"]
    research: str = state["research"]
    print(f"[critic] challenging the claim and research...")

    # The critic sees only the claim and the research — its job is to push back
    prompt: str = (
        f"You are a skeptical critic. Your job is to identify weaknesses, "
        f"counterarguments, and important caveats for the following claim and research. "
        f"Do not agree — find problems.\n\n"
        f"Claim: {claim}\n\nResearch summary:\n{research}\n\nCritique:"
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
    )
    critique: str = (response.choices[0].message.content or "").strip()
    print(f"[critic] done ({len(critique)} chars)")
    return {"critique": critique}

# Verdict node: balance research and critique into a final assessment
def verdict_node(state: SpecializedState) -> dict:
    claim: str = state["claim"]
    research: str = state["research"]
    critique: str = state["critique"]
    print(f"[verdict] synthesizing research and critique...")

    prompt: str = (
        f"You are a balanced analyst. Given the research findings and the critique, "
        f"write a fair final assessment of the claim. Acknowledge both supporting "
        f"evidence and valid objections.\n\n"
        f"Claim: {claim}\n\n"
        f"Research:\n{research}\n\n"
        f"Critique:\n{critique}\n\n"
        f"Final assessment:"
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    verdict: str = (response.choices[0].message.content or "").strip()
    print(f"[verdict] done ({len(verdict)} chars)")
    return {"verdict": verdict}

print("Specialized nodes defined: researcher, critic, verdict.")

<details>
<summary>Details: <strong>Specialization through prompting</strong></summary>

The researcher, critic, and verdict nodes all call the same `gpt-4o-mini` model. The specialization is entirely in the system prompt.

This is how most "specialized agents" work in practice — not different models, but different instructions to the same model. The role is defined by the prompt, not the architecture.

When does this break down? When the required specialization is deep domain knowledge, not just a different perspective. A "legal reasoning" agent and a "code generation" agent would genuinely benefit from fine-tuned or domain-specific models. Role-via-prompt works when the task is within the base model's capability; it fails when the task requires knowledge the base model doesn't have.

Also notice: the critic never searches. It can only challenge based on the research it receives. This is deliberate — we wanted a role that reasons, not retrieves. This is an architectural decision embedded in the node design.
</details>

In [ ]:
# Assemble the specialized-worker graph
spec_builder: StateGraph = StateGraph(SpecializedState)

# Register the three specialized nodes
spec_builder.add_node("researcher", researcher_node)
spec_builder.add_node("critic", critic_node)
spec_builder.add_node("verdict", verdict_node)

# researcher must run before critic (critic reads research)
spec_builder.set_entry_point("researcher")
spec_builder.add_edge("researcher", "critic")
spec_builder.add_edge("critic", "verdict")
spec_builder.add_edge("verdict", END)

specialized_graph = spec_builder.compile()
display(Image(specialized_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Run the specialized system on a contested claim
CLAIM: str = "Neural scaling laws mean that making models larger will continue to improve intelligence indefinitely."

spec_result: SpecializedState = specialized_graph.invoke({
    "claim": CLAIM,
    "research": "",
    "critique": "",
    "verdict": "",
})

print("\n--- Research ---")
print(spec_result["research"])
print("\n--- Critique ---")
print(spec_result["critique"])
print("\n--- Verdict ---")
print(spec_result["verdict"])

---
# Part 9 — Honest Limitations

We have built a working multi-agent system. Before closing, we should be honest about what it is and what it isn't. Agents are powerful but fragile. The field is moving fast and the abstractions are still settling.

In [ ]:
# Deliberately break the orchestrator to show what failure looks like
print("=== Demonstrating failure: ambiguous goal ===\n")

ambiguous_state: MultiAgentState = {
    "goal": "make it better",   # no context — what is 'it'?
    "tasks": [],
    "results": [],
    "final_answer": "",
}

try:
    broken_result: MultiAgentState = multi_agent_graph.invoke(ambiguous_state)
    print("Orchestrator produced tasks anyway:")
    for task in broken_result["tasks"]:
        print(f"  - {task}")
    print()
    print("Observation: the orchestrator hallucinated a context.")
    print("No error was raised. The system appeared to work.")
    print("This is a silent failure — the most dangerous kind.")
except Exception as e:
    print(f"Error: {e}")

<details>
<summary>Details: <strong>Five limitations to know before deploying agents</strong></summary>

**1. Silent failures.** An agent never refuses a badly-specified task. It hallucinate context, invents assumptions, and returns a confident-sounding answer. There is no runtime exception for "I didn't understand the goal."

**2. Error propagation.** In a pipeline, each node's output becomes the next node's input. A bad orchestrator decomposition produces bad worker tasks, which produce bad results, which produce a bad synthesis. The error compounds silently through every stage.

**3. No shared memory between agents.** Each node call is stateless except for what's explicitly in the state dict. If a worker discovers something important that wasn't anticipated as a state field, it cannot communicate it to other workers.

**4. Cost and latency multiply.** Every LLM call adds latency and cost. A 3-task system with search makes at minimum 5 model calls (1 orchestrator + 3 workers + 1 synthesizer) and 3 Tavily calls. Complex multi-agent systems can easily spend 10× what a single-agent solution would.

**5. Prompt brittleness compounds.** Each agent has its own prompt. A small wording change in the orchestrator prompt can change the decomposition, which changes what the workers receive, which changes the synthesis. The system has no integration tests for this.

These are not reasons to avoid agents — they are reasons to build them carefully, add evaluation, and maintain human oversight for high-stakes tasks.
</details>

---
# Recap

| Concept | Where it appeared |
|---|---|
| Why multiple agents | Part 1 — role conflict, context exhaustion, serial bottleneck |
| Orchestrator / worker pattern | Parts 2–6 — decompose → execute → synthesize |
| Language as the agent interface | Part 7 — `str` is the only type crossing agent boundaries |
| Annotated reducers | Part 2 — `operator.add` for accumulating worker results |
| Specialized workers | Part 8 — same model, different prompts, different roles |
| Silent failure | Part 9 — ambiguous goal produces confident hallucination |

**The course payoff:**

In Lecture 1 we looked at a JSON string carrying a tool-call and said: *"This is the interface between the model and the tool."*

Today we looked at a plain English sentence in a list and said: *"This is the interface between the orchestrator and the worker."*

Four lectures later, the string is still the interface. In classical software, interfaces are enforced by type systems. In agent systems, the interface is natural language — and the contract is in the prompt.

---

*The field is evolving rapidly. The frameworks will change. The models will improve. But this fundamental tension — flexibility through language versus reliability through types — will be with us for a long time.*